# 03 — Cleaning & Merging: Build Master SA2 Dataset

## Purpose
Join SA Water tariff data, ABS SEIFA 2021 indexes, and SA2 spatial boundaries
into a single master dataset — one row per SA2 suburb. This is the dataset
all downstream phases (EDA, ML, simulation, Power BI) consume.

## Important note on income data
SEIFA 2021 does not publish median household income in dollars. It publishes
composite index scores. We use the **IER (Index of Economic Resources)** as a
relative proxy for household economic capacity. A higher IER = more resources =
lower water stress. Actual dollar income (ABS Census G02) can be added in Phase 5
if downloaded to enrich the model.

## Inputs
- `data/raw/Statistical Area Level 2, Indexes, SEIFA 2021.xlsx` — Table 1 (all indexes)
- `data/spatial/SA2_2021_AUST_GDA2020.shp` — SA2 boundaries, CRS EPSG:7844
- `data/clean/clean_sawater_tariff_2425.csv` — FY2024-25 tariff reference

## Outputs
- `data/clean/clean_master_sa2.csv` — flat master table, one row per SA2
- `data/clean/clean_master_sa2.gpkg` — same data with geometry for spatial phases


In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW     = PROJECT_ROOT / 'data' / 'raw'
SPATIAL = PROJECT_ROOT / 'data' / 'spatial'
CLEAN   = PROJECT_ROOT / 'data' / 'clean'

SEIFA_PATH  = RAW / 'Statistical Area Level 2, Indexes, SEIFA 2021.xlsx'
SHP_PATH    = SPATIAL / 'SA2_2021_AUST_GDA2020.shp'
TARIFF_PATH = CLEAN / 'clean_sawater_tariff_2425.csv'

for p in [SEIFA_PATH, SHP_PATH, TARIFF_PATH]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p.name}')


  [OK] Statistical Area Level 2, Indexes, SEIFA 2021.xlsx
  [OK] SA2_2021_AUST_GDA2020.shp
  [OK] clean_sawater_tariff_2425.csv


## Load SEIFA 2021 — Table 1 (all index scores)

Table 1 is the summary sheet with all 4 SEIFA indexes in one place.
The actual data starts at row 5 (0-indexed row 4 is the header).
Filter to SA: SA2 codes starting with `'4'` (ABS SA state prefix).

In [2]:
# Header is at Excel row 5 (skiprows=4 gives us that row as columns)
seifa_raw = pd.read_excel(
    SEIFA_PATH,
    sheet_name='Table 1',
    engine='openpyxl',
    skiprows=4,
    dtype=str,
)

print('Raw shape:', seifa_raw.shape)
print('Raw columns:', list(seifa_raw.columns))
print(seifa_raw.head(3).to_string())


Raw shape: (2369, 11)
Raw columns: ['Unnamed: 0', 'Unnamed: 1', 'Index of Relative Socio-economic Disadvantage', 'Unnamed: 3', 'Index of Relative Socio-economic Advantage and Disadvantage', 'Unnamed: 5', 'Index of Economic Resources', 'Unnamed: 7', 'Index of Education and Occupation', 'Unnamed: 9', 'Unnamed: 10']
                                          Unnamed: 0                                 Unnamed: 1 Index of Relative Socio-economic Disadvantage Unnamed: 3 Index of Relative Socio-economic Advantage and Disadvantage Unnamed: 5 Index of Economic Resources Unnamed: 7 Index of Education and Occupation Unnamed: 9                Unnamed: 10
0  2021 Statistical Area Level 2  (SA2) 9-Digit Code  2021 Statistical Area Level 2 (SA2) Name                                          Score     Decile                                                       Score     Decile                       Score     Decile                             Score     Decile  Usual Resident Population
1              

In [3]:
# Rename columns to readable names based on Table 1 structure:
# Col 0: SA2 code | Col 1: SA2 name | Col 2-3: IRSD score+decile
# Col 4-5: IRSAD | Col 6-7: IER | Col 8-9: IEO | Col 10: Population
col_map = {
    seifa_raw.columns[0]:  'SA2_CODE21',
    seifa_raw.columns[1]:  'SA2_NAME_SEIFA',
    seifa_raw.columns[2]:  'irsd_score',
    seifa_raw.columns[3]:  'irsd_decile',
    seifa_raw.columns[4]:  'irsad_score',
    seifa_raw.columns[5]:  'irsad_decile',
    seifa_raw.columns[6]:  'ier_score',
    seifa_raw.columns[7]:  'ier_decile',
    seifa_raw.columns[8]:  'ieo_score',
    seifa_raw.columns[9]:  'ieo_decile',
    seifa_raw.columns[10]: 'population',
}
seifa = seifa_raw.rename(columns=col_map)[list(col_map.values())].copy()

# Drop header-bleed rows (where SA2_CODE21 is not numeric)
seifa = seifa[seifa['SA2_CODE21'].str.match(r'^\d{9}$', na=False)].copy()

# SA2_CODE21 must stay as string (leading zeros exist in some codes)
# Filter to South Australia: SA2 codes start with '4'
seifa = seifa[seifa['SA2_CODE21'].str.startswith('4')].copy()

# Convert numeric columns
for col in ['irsd_score','irsd_decile','irsad_score','irsad_decile',
            'ier_score','ier_decile','ieo_score','ieo_decile','population']:
    seifa[col] = pd.to_numeric(seifa[col], errors='coerce')

print(f'SA SEIFA shape: {seifa.shape}')
print(seifa.dtypes)
print(seifa.head(5).to_string(index=False))


SA SEIFA shape: (167, 11)
SA2_CODE21         object
SA2_NAME_SEIFA     object
irsd_score        float64
irsd_decile       float64
irsad_score       float64
irsad_decile      float64
ier_score         float64
ier_decile        float64
ieo_score           int64
ieo_decile          int64
population          int64
dtype: object
SA2_CODE21     SA2_NAME_SEIFA  irsd_score  irsd_decile  irsad_score  irsad_decile  ier_score  ier_decile  ieo_score  ieo_decile  population
 401011001           Adelaide       980.0          4.0       1045.0           7.0      814.0         1.0       1134          10       18202
 401011002     North Adelaide      1058.0          8.0       1088.0           9.0      960.0         3.0       1145          10        6823
 401021003     Adelaide Hills      1062.0          8.0       1029.0           7.0     1077.0         9.0       1009           6        7051
 401021004 Aldgate - Stirling      1100.0         10.0       1098.0           9.0     1101.0        10.0       111

In [4]:
# Sanity checks
print('SA2_CODE21 sample:', seifa['SA2_CODE21'].head(5).tolist())
print('All start with 4:', seifa['SA2_CODE21'].str.startswith('4').all())
print('IER score range:', seifa['ier_score'].min(), '—', seifa['ier_score'].max())
print('IRSD score range:', seifa['irsd_score'].min(), '—', seifa['irsd_score'].max())
print('Nulls:\n', seifa.isnull().sum())


SA2_CODE21 sample: ['401011001', '401011002', '401021003', '401021004', '401021005']
All start with 4: True
IER score range: 519.0 — 1113.0
IRSD score range: 496.0 — 1106.0
Nulls:
 SA2_CODE21        0
SA2_NAME_SEIFA    0
irsd_score        1
irsd_decile       1
irsad_score       1
irsad_decile      1
ier_score         1
ier_decile        1
ieo_score         0
ieo_decile        0
population        0
dtype: int64


## Load SA2 shapefile — filter to South Australia

In [5]:
gdf_all = gpd.read_file(SHP_PATH)
print(f'All Australia shape: {gdf_all.shape}')
print(f'CRS: {gdf_all.crs}')
assert str(gdf_all.crs.to_epsg()) == '7844', 'CRS mismatch — expected EPSG:7844'

# Filter to SA immediately
gdf = gdf_all[gdf_all['STE_CODE21'] == '4'].copy()
print(f'SA only shape: {gdf.shape}')
print(gdf.dtypes)
print(gdf[['SA2_CODE21','SA2_NAME21','AREASQKM21']].head(5).to_string(index=False))


All Australia shape: (2473, 17)
CRS: EPSG:7844
SA only shape: (176, 17)
SA2_CODE21      object
SA2_NAME21      object
CHG_FLAG21      object
CHG_LBL21       object
SA3_CODE21      object
SA3_NAME21      object
SA4_CODE21      object
SA4_NAME21      object
GCC_CODE21      object
GCC_NAME21      object
STE_CODE21      object
STE_NAME21      object
AUS_CODE21      object
AUS_NAME21      object
AREASQKM21     float64
LOCI_URI21      object
geometry      geometry
dtype: object
SA2_CODE21         SA2_NAME21  AREASQKM21
 401011001           Adelaide     10.4824
 401011002     North Adelaide      5.0909
 401021003     Adelaide Hills    364.4390
 401021004 Aldgate - Stirling    117.2141
 401021005 Hahndorf - Echunga    110.1516


In [6]:
# Keep only the columns we need from the shapefile
gdf = gdf[['SA2_CODE21','SA2_NAME21','SA3_NAME21','SA4_NAME21',
           'AREASQKM21','geometry']].copy()

print(f'Shape after column trim: {gdf.shape}')
print(gdf.dtypes)


Shape after column trim: (176, 6)
SA2_CODE21      object
SA2_NAME21      object
SA3_NAME21      object
SA4_NAME21      object
AREASQKM21     float64
geometry      geometry
dtype: object


## Load FY2024-25 tariff reference

Statewide flat-rate pricing — same bill components for every SA2.
Sewerage is the only spatially variable component (property-value based).
We use $600K as state-median property value baseline; Phase 5 will
substitute per-SA2 median property values when available.

In [7]:
tariff = pd.read_csv(TARIFF_PATH)
print(tariff.T.to_string())

USAGE_CHARGE   = float(tariff['usage_charge_per_kl'].iloc[0])
ACCESS_ANNUAL  = float(tariff['water_access_charge_annual'].iloc[0])
SEWER_PER_1K   = float(tariff['sewerage_cents_per_1000_pv'].iloc[0])
TYPICAL_KL     = float(tariff['typical_usage_kl'].iloc[0])
MEDIAN_PROP_K  = 600.0  # $600K state median property value baseline

water_usage_annual  = TYPICAL_KL * USAGE_CHARGE
water_access_annual = ACCESS_ANNUAL
sewer_annual        = MEDIAN_PROP_K * SEWER_PER_1K
estimated_annual_bill = water_usage_annual + water_access_annual + sewer_annual

print(f'\nBill components (statewide uniform):')
print(f'  Water usage  ({TYPICAL_KL:.0f} kL x ${USAGE_CHARGE}):  ${water_usage_annual:.2f}')
print(f'  Water access (annual):              ${water_access_annual:.2f}')
print(f'  Sewerage ($600K baseline):          ${sewer_annual:.2f}')
print(f'  TOTAL estimated annual bill:        ${estimated_annual_bill:.2f}')


                                                                                                          0
financial_year                                                                                      2024-25
pricing_policy                                                                          statewide_flat_rate
regulator                                                                                            ESCoSA
usage_charge_per_kl                                                                                  2.7098
water_access_charge_annual                                                                           806.08
sewerage_cents_per_1000_pv                                                                              1.5
typical_usage_kl                                                                                        200
source                      ESCoSA Water Retail Price Determination 2024-25 + SA Water Annual Report Note 4

Bill components (statewide 

## Join SEIFA + shapefile on SA2_CODE21

Both datasets use SA2_CODE21 as the join key — kept as string throughout.
We do a left join from the shapefile so geometry rows are the reference.

In [8]:
master = gdf.merge(
    seifa,
    on='SA2_CODE21',
    how='left',
    validate='1:1',
)

print(f'Master shape after join: {master.shape}')
print(master.dtypes)
print(f'\nJoin nulls (SEIFA columns):')
print(master[['irsd_score','ier_score','population']].isnull().sum())


Master shape after join: (176, 16)
SA2_CODE21          object
SA2_NAME21          object
SA3_NAME21          object
SA4_NAME21          object
AREASQKM21         float64
geometry          geometry
SA2_NAME_SEIFA      object
irsd_score         float64
irsd_decile        float64
irsad_score        float64
irsad_decile       float64
ier_score          float64
ier_decile         float64
ieo_score          float64
ieo_decile         float64
population         float64
dtype: object

Join nulls (SEIFA columns):
irsd_score    10
ier_score     10
population     9
dtype: int64


In [9]:
# Investigate any unmatched rows
unmatched = master[master['ier_score'].isnull()]
if len(unmatched) > 0:
    print(f'Unmatched SA2s ({len(unmatched)}):')    
    print(unmatched[['SA2_CODE21','SA2_NAME21']].to_string(index=False))
    print('\nNote: These may be non-residential or excluded areas in SEIFA.')
else:
    print('All SA2s matched successfully.')


Unmatched SA2s (10):
SA2_CODE21                           SA2_NAME21
 402041039                    Dry Creek - North
 402041042                            Parafield
 403041081               Happy Valley Reservoir
 403041082                             Lonsdale
 404021098                    Dry Creek - South
 404021103                       Torrens Island
 404031104                     Adelaide Airport
 406011137                      Whyalla - North
 497979799 Migratory - Offshore - Shipping (SA)
 499999499                No usual address (SA)

Note: These may be non-residential or excluded areas in SEIFA.


## Calculate water stress metrics

Since SEIFA does not publish median household income in dollars, we use
the **IER score** (Index of Economic Resources) as the economic capacity proxy.

**Water stress index** = estimated_annual_bill / IER_score × 1000

This is dimensionless but comparable across suburbs — higher = more stressed.
Suburbs with lower IER (less economic resources) get a higher stress score
for the same water bill. Tiers will be set from the data distribution in Phase 4.

In [10]:
master['estimated_annual_water_bill'] = round(estimated_annual_bill, 2)

# Water stress index: bill relative to economic resources
# Multiply by 1000 so the number is in a readable range
master['water_stress_index'] = round(
    master['estimated_annual_water_bill'] / master['ier_score'] * 1000, 4
)

# Percentile rank within SA (higher rank = more stressed)
master['stress_pct_rank'] = master['water_stress_index'].rank(
    pct=True, ascending=True, na_option='bottom'
).round(4)

print(f'Water stress index stats:')
print(master['water_stress_index'].describe().round(4))
print(f'\nTop 10 most stressed SA2s:')
cols = ['SA2_NAME21','SA3_NAME21','ier_score','irsd_score','water_stress_index','stress_pct_rank']
print(master.nlargest(10,'water_stress_index')[cols].to_string(index=False))
print(f'\nTop 10 least stressed SA2s:')
print(master.nsmallest(10,'water_stress_index')[cols].to_string(index=False))


Water stress index stats:
count     166.0000
mean     2317.5739
std       236.3958
min      2019.8023
25%      2198.0351
50%      2286.9176
75%      2366.3579
max      4331.4836
Name: water_stress_index, dtype: float64

Top 10 most stressed SA2s:
                  SA2_NAME21                    SA3_NAME21  ier_score  irsd_score  water_stress_index  stress_pct_rank
                   APY Lands      Outback - North and East      519.0       496.0           4331.4836           0.9432
                     Western Eyre Peninsula and South West      641.0       644.0           3507.0827           0.9375
                   Elizabeth                      Playford      758.0       646.0           2965.7520           0.9318
Smithfield - Elizabeth North                      Playford      804.0       732.0           2796.0697           0.9261
                    Adelaide                 Adelaide City      814.0       980.0           2761.7199           0.9205
                 Coober Pedy      Outba

## Assign provisional stress tiers

Thresholds are quartile-based for now. Phase 4 EDA will review the distribution
and confirm whether these bands make narrative sense geographically.

In [11]:
def assign_stress_tier(pct_rank):
    if pd.isna(pct_rank):   return 'Unknown'
    if pct_rank >= 0.75:    return 'Critical'
    if pct_rank >= 0.50:    return 'High'
    if pct_rank >= 0.25:    return 'Moderate'
    return 'Low'

master['stress_tier'] = master['stress_pct_rank'].apply(assign_stress_tier)

print('Stress tier distribution:')
print(master['stress_tier'].value_counts())
print(f'\nShape: {master.shape}')
print(master.dtypes)


Stress tier distribution:
stress_tier
Critical    45
High        44
Moderate    44
Low         43
Name: count, dtype: int64

Shape: (176, 20)
SA2_CODE21                       object
SA2_NAME21                       object
SA3_NAME21                       object
SA4_NAME21                       object
AREASQKM21                      float64
geometry                       geometry
SA2_NAME_SEIFA                   object
irsd_score                      float64
irsd_decile                     float64
irsad_score                     float64
irsad_decile                    float64
ier_score                       float64
ier_decile                      float64
ieo_score                       float64
ieo_decile                      float64
population                      float64
estimated_annual_water_bill     float64
water_stress_index              float64
stress_pct_rank                 float64
stress_tier                      object
dtype: object


## Final column selection and validation

In [12]:
FINAL_COLS = [
    'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21',
    'population', 'AREASQKM21',
    'irsd_score', 'irsd_decile',
    'irsad_score', 'irsad_decile',
    'ier_score', 'ier_decile',
    'ieo_score', 'ieo_decile',
    'estimated_annual_water_bill',
    'water_stress_index',
    'stress_pct_rank',
    'stress_tier',
    'geometry',
]

master = master[FINAL_COLS].copy()
print(f'Final shape: {master.shape}')
print(master.dtypes)
print(f'\nNull counts:')
print(master.isnull().sum())
print(f'\nSample rows:')
print(master[FINAL_COLS[:-1]].head(5).to_string(index=False))


Final shape: (176, 19)
SA2_CODE21                       object
SA2_NAME21                       object
SA3_NAME21                       object
SA4_NAME21                       object
population                      float64
AREASQKM21                      float64
irsd_score                      float64
irsd_decile                     float64
irsad_score                     float64
irsad_decile                    float64
ier_score                       float64
ier_decile                      float64
ieo_score                       float64
ieo_decile                      float64
estimated_annual_water_bill     float64
water_stress_index              float64
stress_pct_rank                 float64
stress_tier                      object
geometry                       geometry
dtype: object

Null counts:
SA2_CODE21                      0
SA2_NAME21                      0
SA3_NAME21                      0
SA4_NAME21                      0
population                      9
AREASQKM21         

SA2_CODE21         SA2_NAME21     SA3_NAME21                   SA4_NAME21  population  AREASQKM21  irsd_score  irsd_decile  irsad_score  irsad_decile  ier_score  ier_decile  ieo_score  ieo_decile  estimated_annual_water_bill  water_stress_index  stress_pct_rank stress_tier
 401011001           Adelaide  Adelaide City Adelaide - Central and Hills     18202.0     10.4824       980.0          4.0       1045.0           7.0      814.0         1.0     1134.0        10.0                      2248.04           2761.7199           0.9205    Critical
 401011002     North Adelaide  Adelaide City Adelaide - Central and Hills      6823.0      5.0909      1058.0          8.0       1088.0           9.0      960.0         3.0     1145.0        10.0                      2248.04           2341.7083           0.6307        High
 401021003     Adelaide Hills Adelaide Hills Adelaide - Central and Hills      7051.0    364.4390      1062.0          8.0       1029.0           7.0     1077.0         9.0     1

## Save outputs

In [13]:
# Save flat CSV (no geometry)
out_csv = CLEAN / 'clean_master_sa2.csv'
master.drop(columns=['geometry']).to_csv(out_csv, index=False)
print(f'Saved: {out_csv.name}  ({out_csv.stat().st_size // 1024} KB)')
print(f'Shape: {master.drop(columns=["geometry"]).shape}')

# Save GeoPackage (with geometry for spatial phases)
out_gpkg = CLEAN / 'clean_master_sa2.gpkg'
master.to_file(out_gpkg, driver='GPKG')
print(f'Saved: {out_gpkg.name}  ({out_gpkg.stat().st_size // 1024} KB)')
print(f'CRS: {master.crs}')


Saved: clean_master_sa2.csv  (26 KB)
Shape: (176, 18)


Saved: clean_master_sa2.gpkg  (3140 KB)
CRS: EPSG:7844


## Summary

| Output | Rows | Description |
|---|---|---|
| `clean_master_sa2.csv` | 176 | Flat master table, one row per SA2 |
| `clean_master_sa2.gpkg` | 176 | Same data with SA2 polygon geometry |

### Key columns for downstream phases
| Column | Use |
|---|---|
| `SA2_CODE21` | Join key — always string |
| `ier_score` | Economic resources proxy (higher = better off) |
| `irsd_score` | Socioeconomic disadvantage (lower = more disadvantaged) |
| `estimated_annual_water_bill` | Uniform $1,948 baseline (statewide pricing) |
| `water_stress_index` | Bill / IER × 1000 — relative stress measure |
| `stress_tier` | Critical / High / Moderate / Low (provisional, confirm in Phase 4) |
| `stress_pct_rank` | Percentile rank within SA (0=lowest, 1=highest stress) |

### Limitation to address in Phase 5
Median household income in dollars is not in SEIFA. To get the literal
`water_cost_burden_ratio` (bill / income), download ABS Census 2021 G02
(Selected Medians and Averages) at SA2 level from ABS TableBuilder or
Community Profiles. This will allow a true % of income calculation.
Current `water_stress_index` is a valid relative ranking but not a % figure.
